In [ ]:
import pandas as pd
import networkx as nx

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Graph"
u_col, v_col, w_col = "u", "v", "w"
df = pd.read_excel(file_path, sheet_name=sheet_name)

# ========= 2) 参数模板 =========
params = {
    "source": 1,      # 起点
    "target": 5,      # 终点
    "directed": True  # 是否有向图
}

G = nx.DiGraph() if params["directed"] else nx.Graph()
for _, row in df.iterrows():
    G.add_edge(row[u_col], row[v_col], weight=float(row[w_col]))

dist = nx.dijkstra_path_length(G, params["source"], params["target"], weight="weight")
path = nx.dijkstra_path(G, params["source"], params["target"], weight="weight")
print(dist, path)


In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
# 方式1：Excel中直接放邻接矩阵（含表头），读取后转numpy
file_path = r"your_data.xlsx"
sheet_name = "AdjMatrix"
df = pd.read_excel(file_path, sheet_name=sheet_name, index_col=0)
adj = df.to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {
    "inf_value": 1e9  # 用于表示不可达的大数（若你数据不是np.inf）
}

adj[adj >= params["inf_value"]] = np.inf
np.fill_diagonal(adj, 0)

dist = adj.copy()
n = dist.shape[0]
for k in range(n):
    for i in range(n):
        for j in range(n):
            dist[i, j] = min(dist[i, j], dist[i, k] + dist[k, j])

print(dist)


In [ ]:
"""
Dijkstra、Floyd

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "Dijkstra、Floyd.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
FROM_COLUMN = "起点"  # TODO: 请填写[起点列名]，说明：边起点。
TO_COLUMN = "终点"  # TODO: 请填写[终点列名]，说明：边终点。
WEIGHT_COLUMN = "权重"  # TODO: 请填写[权重列名]，说明：距离、时间或成本，Dijkstra 要求非负。
SOURCE_NODE = "A"  # TODO: 请填写[源点]，说明：路径起点。
TARGET_NODE = "D"  # TODO: 请填写[终点]，说明：路径终点。
DIRECTED = False  # TODO: 请填写[是否有向图]，说明：True 表示有向边，False 表示无向边。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    G = nx.DiGraph() if DIRECTED else nx.Graph()
    for _, row in data.iterrows():
        G.add_edge(row[FROM_COLUMN], row[TO_COLUMN], weight=float(row[WEIGHT_COLUMN]))
    path = nx.dijkstra_path(G, SOURCE_NODE, TARGET_NODE, weight="weight")
    distance = nx.dijkstra_path_length(G, SOURCE_NODE, TARGET_NODE, weight="weight")
    floyd = dict(nx.floyd_warshall(G, weight="weight"))
    pd.DataFrame({"最短路径": [" -> ".join(map(str, path))], "距离": [distance]}).to_csv(
        OUTPUT_FILE, index=False, encoding="utf-8-sig"
    )
    print("Dijkstra 最短路径:", path, "距离:", distance)
    print("Floyd 任意两点距离示例:", floyd[SOURCE_NODE])


if __name__ == "__main__":
    df = load_data()
    run_model(df)
